# CS3DM - Lab 5

## Objectives
- Understand the basic aspects of machine learning (in the context of data mining) using _Scikit-learn_
- Apply your knowledge of _Pandas_ (Lab 3) and _Scikit-learn_ (Lab 4) to solve a simple classification task.

## Suggested Reading
Jake VandePlas, *Python Data Science Handbook*, Chapter 5 (Machine Learning): <https://jakevdp.github.io/PythonDataScienceHandbook/>

## Instructions
This is a follow-on tutorial, with some _**TO DO**_ blocks containing specific activities. Please take your time at each block to understand what is being done, and feel free to tinker, explore and modify any block to check your understanding (if you break anything you can always ctrl-Z, or in the worst case download the original Notebook again).
*****

## Machine learning (in the context of data mining)

Fundamentally, machine learning (ML for short) involves building mathematical models to help understand data. _Greater machine learning_ can be considered as encompassing classical statistical modelling approaches such as linear regression, as well as black box nonlinear models such as artificial neural networks. The _learning_ part comes from the fact that these models have parameters whose best values need to be determined based on the available data - i.e., the model "learns" from the data. Once these models have been fit to some existing data, they can be used to predict and understand aspects of new, not-yet-seen data.

In the context of Data Mining, ML is often used in the **modelling** block of the data mining pipeline. There are two broad areas of modelling that are usually tackled by ML:

- Supervised learning, which includes classification and regression, involves modeling the relationship between measured features of data and some dependent variable (e.g., a label or a numeric value) associated with each observation. Once the model is determined, it can be used to attribute labels/values to new, unknown data.

- Unsupervised learning, which includes clustering, outlier detection and association rule mining, involves modeling the features of a dataset without reference to any label/value, and is often described as "_letting the dataset speak for itself_".

Besides these two, there are other modes that are often discussed, e.g., _semi-supervised learning_, _self-supervised learning_, etc.


### Machine learning is not magic
The term _machine learning_ is sometimes thrown around as if it is some kind of magic pill: apply machine learning to your data, and all your problems will be solved! 

While machine learning provides a set of powerful methods for data mining, it can only go so far without the input of subject-specific knowledge. This can be better understood if you think about ML as a set of methods capable of helping you answer questions about your data: if you don't know what the questions are, then you cannot know if the answers make sense or not. Or, to quote George Harrison's song, _if you don't know where you're going, any road can take you there._
    
Even if the whole "knowing which questions make sense" bit is well covered, machine learning will still not be magical. John Tukey, which is arguably the most influential early pioneer in Data Science, had a [famous aphorism](https://www.jstor.org/stable/2683137) about it: "_The combination of some data and an aching desire for an answer does not ensure that a reasonable answer can be extracted from a given body of data_". This can be read in two (not mutually exclusive) ways:

- The required information to answer a particular question may not be contained in the data. The data may be noisy, incomplete, biased, or unrepresentative of the phenomenon of interest.
    
- The model used may be unable to capture the structure of the relationship between the variables of interest. This can happen, for instance. if the model is too simple or was set with an inadequate structure. An extreme example can be seen if we try to use a simple linear regression to model a clearly nonlinear relationship:

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib as mpl                           # matplotlib and seaborn are two of the most common data visualisation 
import seaborn as sns                              # packages in Python
import matplotlib.pyplot as plt                    # Notice that we can import specific functions and givem them an alias
from sklearn.linear_model import LinearRegression

# Set the default seaborn graphical theme
sns.set()

# Simulate a (slightly noisy) quadratic relationship between two variables x and y
rng = np.random.RandomState(0)                      # Set the seed of the random number generator, for reproducibility
x = np.linspace(-5, 5, 50)[:, np.newaxis]
y = (x ** 2) + .5 * rng.randn(50)[:, np.newaxis]
y = y - np.mean(y)

# Fit a linear regression of the form y = b*x + a
regr  = LinearRegression() # inintialise a linear regression model
model = regr.fit(x, y)     # fit the model

# extract model coefficients
a = model.intercept_       
b = model.coef_

# Plot data + resulting trend line
yp = a * x + b
fig = plt.figure(figsize = (10,6))
plt.plot(x, y, 'ok', alpha = 0.5)
plt.plot(x, (x ** 2 - np.mean(x ** 2)), '--b')
plt.plot(x, yp, ':r', linewidth = 3);

Notice here that although the variables x and y have a clear relationship, the model structure is unable to capture it - in fact the **optimal** (in the mathematical sense) estimate under the model structure is $y = 0~~\forall x\in[-5,5]$: the _null model_, which is clearly not optimal in terms of utility.

Another problem can happen on the opposite end of the spectrum: models that are too complex can also fail to provide relevant answers, because they may _overfit_ the data - i.e., the capture not only the real relationships between the variables of interest, but also any noise that may be present.

*****

# The Scikit-learn library

There are several Python libraries which provide solid implementations of a range of machine learning algorithms. One of the best known is _Scikit-learn_, a package that provides efficient versions of a large number of common algorithms. This package was originally released in 2007, and has steadily grown both in terms of usability and diversity of models provided.

_Scikit-learn_ is characterized by a clean, uniform, and streamlined API, as well as by very useful and complete online documentation. A benefit of this uniformity is that once you understand the basic use and syntax of _Scikit-learn_ for one type of model, switching to a new model or algorithm is very straightforward.


## Data representation in Scikit-learn

- _Scikit-learn_ operates in terms of data tables, which are rectangular data representations similar to the classic dataframes. For instance, if we load the _iris_ dataset that comes in the _Seaborn_ library, we can see the data table structure (which, at this point, should be absolutely familiar to you):

In [ ]:
iris = sns.load_dataset('iris')
iris.head(10) # first 10 rows of the data

Assuming that we want, e.g., to be able to predict the value of `species` for any new sample that may be collected in the future, based on the observed values of the other attributes, we can split this table into two components:

- The _features matrix_, which contains the predictors (in this case the sepal and petal dimensions). This is most often contained in a `NumPy` array or a `Pandas DataFrame`, though some _Scikit-learn_ models also accept `SciPy` sparse matrices.

- The _target array_, which contains (in the particular context of supervised learning - in this case, classification) the quantity (or quantities) we want to be able to predict from the features. This is generally contained in a `NumPy` array or `Pandas Series`.
    
This matrix representation of data is usually performed row-wise: for instance, the features matrix $X$ is a $[n_{samples} \times n_{features}]$ matrix, and the target array a $[n_{samples} \times 1]$ vector (there are a few models that accept more than one column for the target array - which means multiple predicted variables - but we won't be covering these).

## Example of Data Representation for Scikit-learn

Generally speaking, we want out feature matrix and target arrays to have the general shape below (figure source: [Jake VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.02-introducing-scikit-learn.html))

<center><img src = "https://jakevdp.github.io/PythonDataScienceHandbook/figures/05.02-samples-features.png" width = 600></center>

Consider again the _iris_ dataset. To get it in the usual shape expected by the _Scikit-learn_ modelling functions, we just need to extract the features matrix and the target array, which we can do quite easily:

In [ ]:
X_iris = iris.iloc[:, :4] # Remember: we can subset using iloc or...
y_iris = iris['species']  # ... using a variable name 
print(X_iris.shape)
print(y_iris.shape)

## The Scikit-learn API

The _Scikit-learn_ API is designed around the following guiding principles (you can read more [here](https://arxiv.org/abs/1309.0238)):

- Consistency: All objects share a common interface drawn from a limited set of methods, with consistent documentation.
- Inspection: All specified parameter values are exposed as public attributes.
- Limited object hierarchy: Only algorithms are represented by Python classes; datasets are represented in standard formats (`NumPy ndarrays`, `Pandas DataFrames`, `SciPy` sparse matrices) and parameter names use standard Python strings.
- Composition: Many machine learning tasks can be expressed as sequences of more fundamental algorithms, and _Scikit-learn_ makes use of this wherever possible.
- Sensible defaults: When models require user-specified parameters, the library defines an appropriate default value.

Every machine learning algorithm in _Scikit-learn_ is implemented via the _Estimator API_, which provides a consistent interface for a wide range of machine learning applications. The usual steps involved in using the API are as follows:

- Arrange the data into a features matrix and target vector.    
- Choose a class of model by importing the appropriate estimator class from _Scikit-learn_.
- Choose model hyperparameters by instantiating this class with desired values.
- Fit the model to your data by calling the `fit()` method of the model instance.
- Apply the Model to new data:
- For supervised learning, often we predict labels for unknown data using the `predict()` method.
- For unsupervised learning, we often transform or infer properties of the data using the `transform()` or `predict()` methods.


## Simple classification

Let's try using _Scikit-learn_ to build a simple classification model, using the Iris dataset we discussed earlier. Our question will be this: given a model trained on a portion of the Iris data, how well can we predict the remaining labels?

We will use a simple model known as Gaussian Naïve Bayes model, which essentially assumes that all features are independent and normally distributed. Because it is quite fast and has no hyperparameters to choose, Gaussian Naïve Bayes is often a good baseline model to use as a sanity check for benchmarking more sophisticated models.

We generally want to evaluate models on data that was not used for training (more on that later), so we will split the data into a training set and a testing set. This can be easily done by hand, but _Scikit-learn_ already provides a handy shortcut:

In [ ]:
from sklearn.model_selection import train_test_split  # Import the data splitting method
from sklearn.naive_bayes import GaussianNB            # Import the Gaussian NB model
from sklearn.metrics import accuracy_score            # Import accuracy calculation

# Split the data
Xtrain, Xtest, ytrain, ytest = train_test_split(X_iris, y_iris,
                                                random_state = 1, 
                                                test_size = 0.5) # Use 1/2 for training, 1/2 for testing


model = GaussianNB()             # Instantiate model
model.fit(Xtrain, ytrain)        # Fit model to data

# Use model to predict class for the test data
y_model = model.predict(Xtest)   


# Calculate Accuracy of the model on the test data
# (Remember: Accuracy is the fraction of predicted labels that match their true value)
myAcc = accuracy_score(ytest, y_model)
print("Accuracy:", np.round(myAcc, 4))

## Model validation

In principle, model validation is very simple: after choosing a model (and optimising ots hyperparameters - more on that in future labs), we can estimate how effective it is by using it to predict labels for some data and comparing the predictions to the known values. This, however, needs to be done with care.

A fundamental flaw that can occur is to train and evaluate the model on the same data. This represents a risk because you have no way to evaluate the _generalisation_ ability of your model if it is evaluated using the same data to which it was fit. In the most pathological cases you can get an estimated prediction accuracy of 100% that means absolutely nothing in terms of your model's ability to predict _new_ data.

As an example of this pathological behavior, let's look at a kNN model with k = 1, trained and tested on the same data:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier # Select model
model = KNeighborsClassifier(n_neighbors = 1)      # Instantiate model, define hyperparameter k


model.fit(Xtrain, ytrain)

# Evaluate using the same data used for training (NOTE: This is NOT recommended in practice)
ytest_samedata = model.predict(Xtrain)

print("Estimated accuracy on training data:", accuracy_score(ytrain, ytest_samedata))


This suggests 100% accuracy! But is this actually true? In fact, this approach contains that fundamental flaw of training and evaluating the model on the same data. Worst still, the kNN model is an instance-based estimator that simply stores the training data, and predicts labels by comparing new data to these stored points: except in contrived cases, it will get 100% in-sample accuracy every time (regardless of its real generalisation performance)!

One way to prevent this is to use what's known as a _holdout set_, which is exactly what we did earlier with the Naïve Bayes classifier: we hold back some subset of the data from the training of the model, and then use this holdout set to check the model performance.

In [ ]:
# evaluate the model on the second set of data
y2_model = model.predict(Xtest)
print("Estimated accuracy on test set:", np.round(accuracy_score(ytest, y2_model), 4))

One disadvantage of using a holdout set for model validation is that we have lost a portion of our data to the model training. In the above case, half the dataset does not contribute to the training of the model! This cansometimes cause problems, especially if the initial set of training data is small.

One way to address this is to use _k-fold cross-validation_, which is one of the most common methods for evaluating classifiers:

1. Data is split into k equal-sized (ish) subsets, or _folds_.
2. For each fold _i_: train a model on all other data, and use fold _i_ for testing.
3. The performance estimates on each fold are then averaged to yield an overall estimate.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_acc = cross_val_score(model, X_iris, y_iris, 
                         cv = 5) # number of folds. 5 and 10 are common choices

print("5-fold CV accuracy =", 
      np.mean(cv_acc))

*****
## **TO DO**
(30 minutes)

Now it's your turn to try you hand at modelling! We'll use the _Wisconsin Breast Cancer_ dataset as our example (please download from Blackboard), and we want to build a model capable of predicting the type of tumour (benign or malignant) based on a series of measurements.

For that, you'll use the _Pandas_ skills that you have gained from Lab 3, as well as the _Scikit-learn_ knowledge you just acquired.

You task is the following:

- Load the data from the CSV file (check Lab 3 if you don't remember how to load a CSV)
- Check the first rows of the data frame.
- Remove any ID variables.
- Separate the predictor and class columns into two different objects. Call these `X` (for the predictors) and `y` (for the target class).
- Check the presence of missing values in any variables. Check methods `.isnull()` and `.sum()`, which can be called for Pandas dataframes. Notice that they can be chained ;-)
- Replace missing values on each variable by the mean of the non-missing values for that variable. Check function `SimpleImputer`, available under submodule `skelearn.impute`,
- Scale all variables to the [0,1] interval. Check the `MinMaxScaler` function from `sklearn.preprocessing`.
- Check the _class balance_ of this problem. Check method `.value_counts()`, which can be called for Pandas dataframes.
- Split your data into training and test sets. Use 80% for training and 20% for testing. To ensure both splits keep the same balance of the data, make sure that your splitting is stratified by class (this can be done by settion option `stratify = y` in function `train_test_split()`.
- Build a _Random Forest_ classifier using your **training** data.
- Check the accuracy of your model in two different ways:
    - Calculate accuracy based on model predictions for the training set itself (training performance - probably overly optimistic)
    - Calculate accuracy based on model predictions for the test set (better estimate of generalisation performance)

In [ ]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
#load in csv
df_all = pd.read_csv('breastcancer.csv')
#drop ID cols
df_all.drop('Id', axis = 1, inplace = True)
#df_all.head()
#split into class (y) and predictors (x)
df_x = df_all.drop('Class', axis = 1)
df_y = df_all['Class']
print(df_x.shape)
print(df_y.shape)
#x.head()
#y.head()

In [ ]:
#Initialise mean imputer
mean_imputer = SimpleImputer(missing_values = np.nan, strategy = 'mean')
#check x for missing values
x_missing_per_col = df_x.isnull().sum(axis = 0)
print(x_missing_per_col)
#run the imputer on df_x, giving a ndarray of df_x with NANs replaced with the mean value for that column
x_impd = mean_imputer.fit_transform(x)


In [ ]:
#convert the imputed ndarray back to a data frame, check the shape still matches and that there are no missing values.
df_x = pd.DataFrame(x_impd, columns = df_x.columns)
print(df_x.shape)
x_missing_per_col = df_x.isnull().sum(axis = 0)
print(x_missing_per_col)

In [ ]:
#Initialise Scaler
scaler = MinMaxScaler()
#Carry out default 0 - 1 scaling on df_x
x_scaled = scaler.fit_transform(df_x)
print(x_scaled.shape)
# convert back to dataframe
df_x_scaled = pd.DataFrame(x_scaled, columns = df_x.columns)
print(df_x_scaled.shape)
print(df_x_scaled.head())


In [ ]:
#check distribution of class values
print(df_y.value_counts())

In [ ]:
#Split into 80/20 training and testing
x_train, x_test, y_train, y_test = train_test_split(df_x_scaled, df_y, test_size = 0.2, random_state = 1, stratify = df_y)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

#intialise random forest classifier with default params
rfclf = RandomForestClassifier()
#fit the classifier to the training data
rfclf.fit(x_train, y_train)
#score on training data and test data. training data should be 100%, and the test data gives a more accurate representation of how the model performs.
rfclf.score(x_train, y_train)
rfclf.score(x_test, y_test)

# OCD Quiz 2

In [ ]:
import pandas as pd
#load in all csv
df_quiz2 = pd.read_csv("breastcancer.csv")
df_quiz2.head()

In [ ]:
df_quiz2.drop("Id", axis = 1)
df_predictor = df_quiz2.drop("Class", axis = 1)
df_class = df_quiz2[["Class"]].copy()